# 🇱🇰 SynhalEES: Colab Slim-Export Helper
⚡ **Free Colab cloud bandwidth — zero home data cost.**

🔄 **Flow:** Cloud Colab ➔ `pull` (heavy 60MB+ files stay on cloud) ➔ `slim-export` (tiny 1KB CSV) ➔ Download slim CSV ➔ Home `slim-import`.

| Step | Execution | Internet Cost |
|---|---|---|
| 📥 Pull (60MB x N tasks) | Google Colab (Free cloud) | **0 MB (Free)** |
| ⚙️ Slim-Export (extract metrics) | Google Colab | **0 MB (Free)** |
| 💾 Download slim CSV | Colab ➔ Your Computer | **~1 KB only** |
| 📊 Slim-Import & Leaderboard Rebuild | Local terminal | **0 MB (Local)** |


## 🔑 1. Kaggle Login & Environment Setup (Run Once)
Clone the repository and install dependencies, then log in. **Recommended:** put a Kaggle **API token** (kaggle.com/settings/api -> Generate New Token) into the Colab Secret `KAGGLE_API_TOKEN`. Legacy `KAGGLE_USERNAME` + `KAGGLE_KEY` secrets or an uploaded `kaggle.json` also work. The last check verifies real API access.

In [ ]:
# 🔑 Step 1 | Kaggle login & environment setup
!test -d SynhalEES-Benchmark || git clone https://github.com/SynhalaAI/SynhalEES-Benchmark.git
%cd SynhalEES-Benchmark
!git pull --ff-only -q || true  # refresh to latest fixes
!pip install -q -e . "kaggle>=2.2.4"  # pin: Colab preinstalls an old kaggle (only push/run); >=2.2.4 has list/status/download
import json as _json, os as _os
from pathlib import Path as _Path
kg_dir = _Path.home() / '.kaggle'
kg_dir.mkdir(parents=True, exist_ok=True)
kg = kg_dir / 'kaggle.json'
saved, mode = False, None
# Option A (recommended): Colab Secret KAGGLE_API_TOKEN - kaggle.com/settings/api > Generate New Token
try:
    from google.colab import userdata
    _tok = (userdata.get('KAGGLE_API_TOKEN') or '').strip()
    if _tok:
        (kg_dir / 'access_token').write_text(_tok)
        (kg_dir / 'access_token').chmod(0o600)
        saved, mode = True, 'access token (Secret KAGGLE_API_TOKEN)'
        print('kaggle login: API token secret OK')
except Exception as _e:
    print('No KAGGLE_API_TOKEN secret (add one? secrets tab > + Add secret):', _e)
# Option B (legacy): Secrets KAGGLE_USERNAME + KAGGLE_KEY -> kaggle.json
if not saved:
    try:
        from google.colab import userdata
        _u = (userdata.get('KAGGLE_USERNAME') or '').strip()
        _k = (userdata.get('KAGGLE_KEY') or '').strip()
        if _u and _k:
            kg.write_text(_json.dumps({'username': _u, 'key': _k}))
            kg.chmod(0o600)
            saved, mode = True, 'legacy username/key (KAGGLE_USERNAME + KAGGLE_KEY)'
            print('kaggle login: legacy Colab Secrets OK')
    except Exception as _e:
        print('No legacy secrets:', _e)
# Option C (fallback): upload kaggle.json from Kaggle > Settings > API
if not saved:
    try:
        from google.colab import files
        print('Upload kaggle.json (Kaggle > Settings > API > Create New Token)')
        up = files.upload()
        for _name, _data in up.items():
            kg.write_bytes(_data)
            break
        kg.chmod(0o600)
        saved, mode = kg.exists(), 'uploaded kaggle.json'
    except ImportError:
        print('Not on Colab; ensure ~/.kaggle/access_token or ~/.kaggle/kaggle.json exists.')
assert saved, 'kaggle login failed: add Secret KAGGLE_API_TOKEN (recommended) or KAGGLE_USERNAME + KAGGLE_KEY'
print('kaggle login via', mode, '-> ready')
# real auth check: the kaggle CLI turns HTTP 401 into generic help text, so detect it ourselves
import subprocess as _sp, sys as _sys
_r = _sp.run([_sys.executable, '-m', 'kaggle', 'b', 't', 'list'], capture_output=True, text=True, encoding='utf-8', errors='replace')
_out = (_r.stdout or '') + (_r.stderr or '')
if _r.returncode == 0:
    print('OK: kaggle auth works - your benchmark tasks:')
    print('\n'.join(_out.splitlines()[:8]))
else:
    print(_out[:1200])
    if 'Authentication required' in _out or '401' in _out:
        print('X: Kaggle REJECTED the credentials (HTTP 401).')
        print('  -> Recommended: Generate New Token at https://www.kaggle.com/settings/api')
        print('     (single token string) and store it in Colab Secret KAGGLE_API_TOKEN, then re-run.')
        print('  -> Legacy: fix Secret pair KAGGLE_USERNAME + KAGGLE_KEY (exact, no spaces/newlines).')


## 📥 2. Pull Artifacts on Cloud (Zero Home Internet Used)
Set `MODE = 'all'` + `MODALITY` to pull tasks by size group: `text` (15 tasks, ~145MB, default), `vision` (~141MB), `audio` (~590MB), or `all` (~875MB) - download vision/audio separately when you want them. `MODE = 'one'` + `TASK` pulls a single pillar. `MODEL` optionally targets one specific model.

In [ ]:
# 📥 Step 2a | Pull config (MODE / MODALITY / TASK / MODEL)
import os
MODE = 'all'  # 'one' = single task below | 'all' = every task matching MODALITY
MODALITY = 'text'  # MODE='all' filter: 'text' (15 tasks ~145MB) | 'vision' (~141MB) | 'audio' (~590MB) | 'all' (~875MB)
TASK = 'synhalees-05-sinhala-grammar'  # used when MODE='one'
MODEL = ''  # optional: e.g. 'gemini-3.7-flash' (empty = all models)
os.environ['NB_TASK'] = 'all' if MODE.strip().lower() == 'all' else TASK.strip()
os.environ['NB_MODALITY'] = (MODALITY or 'text').strip().lower()
os.environ['NB_MODEL'] = '-m ' + MODEL.strip() if MODEL.strip() else ''
print('pulling:', os.environ['NB_TASK'], '| modality:', os.environ['NB_MODALITY'], '|', MODEL.strip() or '(all models)')


In [ ]:
# 📥 Step 2b | Pull run artifacts from Kaggle
!python -m synhalees kaggle pull "$NB_TASK" --modality "$NB_MODALITY" $NB_MODEL


## 📦 3. Slim-Export Metrics (Convert 60MB to ~1KB)
Extracts accuracy score, cost, tokens, and latency from `result.json` & `run.json` into lightweight `.slim.csv` files - only for the tasks you pulled (`MODALITY` decides).

In [ ]:
# 📦 Step 3 | Slim-export: metrics 60MB -> ~1KB CSV
!if [ "$NB_TASK" = "all" ]; then case "$NB_MODALITY" in vision) _tl="synhalees-vision" ;; audio) _tl="synhalees-audio" ;; all) _tl="synhalees-01-buddhist-culture synhalees-02-pali-gatha synhalees-03-classical-literature synhalees-04-kavi-sindu synhalees-05-sinhala-grammar synhalees-06-daily-spoken synhalees-07-figurative-sinhala synhalees-08-profanity-nuance synhalees-09-singlish-sms synhalees-10-regional-dialects synhalees-11-astrology-beliefs synhalees-12-general-knowledge synhalees-13-sri-lanka-law synhalees-14-culinary-kitchen synhalees-15-numbers-maths synhalees-vision synhalees-audio" ;; *) _tl="synhalees-01-buddhist-culture synhalees-02-pali-gatha synhalees-03-classical-literature synhalees-04-kavi-sindu synhalees-05-sinhala-grammar synhalees-06-daily-spoken synhalees-07-figurative-sinhala synhalees-08-profanity-nuance synhalees-09-singlish-sms synhalees-10-regional-dialects synhalees-11-astrology-beliefs synhalees-12-general-knowledge synhalees-13-sri-lanka-law synhalees-14-culinary-kitchen synhalees-15-numbers-maths" ;; esac; for _t in $_tl; do echo "== $_t =="; python -m synhalees kaggle slim-export "$_t" || true; done; else python -m synhalees kaggle slim-export "$NB_TASK"; fi
!ls -la kaggle-results/*.slim.csv


## 💾 4. Download Lightweight Slim CSVs (Only KBs to Download)
Downloads only the generated tiny CSVs to your machine.

In [ ]:
# 💾 Step 4 | Download slim CSVs to this machine
from google.colab import files
import glob
for f in sorted(glob.glob('kaggle-results/*.slim.csv')):
    print(f)
    files.download(f)


## 🚀 5. Import & Rebuild Leaderboard on Local PC (No Internet Required)
Run this in your local terminal to import the downloaded slim CSVs directly into the leaderboard without re-downloading anything from Kaggle:

```bash
# Import one or all slim CSVs into submissions/ and automatically rebuild leaderboard data
python -m synhalees kaggle slim-import kaggle-results/synhalees-05-sinhala-grammar.slim.csv
```

💡 **Note (AGENTS.md):** The slim CSV strictly mirrors the `EXT_HEADER` schema (`model,provider,date,pillar,modality,score,cost_usd,tokens,latency_ms`). Re-check notebook cells if the schema ever changes.